# 02 — NDVI baseline & two-date clearing

Threshold NDVI to map forest for both study dates and flag newly cleared pixels (forest in date 1, not forest in date 2). Mirrors `src/baseline/ndvi.py` and `src/change_detection.py`.

> Requires `earthengine authenticate` and a Cloud project (`EE_PROJECT`).


In [ ]:
import os, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

import ee
import geemap
import numpy as np
from data.earth_engine import init, get_cloud_free_composite
from data.regions import REGION, DATE1, DATE2, CLOUD_PCT
from baseline.ndvi import compute_ndvi, ndvi_forest_mask, iou

init()
region = ee.Geometry.Rectangle(REGION["bounds"])

In [ ]:
def forest_mask(window, threshold=0.6):
    composite = get_cloud_free_composite(region, *window, CLOUD_PCT)
    ndvi = composite.normalizedDifference(["B8", "B4"]).rename("NDVI")
    return ndvi.gt(threshold), ndvi

forest1, ndvi1 = forest_mask(DATE1)
forest2, ndvi2 = forest_mask(DATE2)

def hectares(mask):
    return mask.multiply(ee.Image.pixelArea()).divide(1e4).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=region, scale=10, maxPixels=1e9,
    ).getInfo()

print("forest ha (date 1, 2020):", hectares(forest1))
print("forest ha (date 2, 2022):", hectares(forest2))

In [ ]:
# Newly cleared = forest at date 1, not forest at date 2
cleared = forest1.And(forest2.Not())
cleared_ha = cleared.multiply(ee.Image.pixelArea()).divide(1e4).reduceRegion(
    reducer=ee.Reducer.sum(), geometry=region, scale=10, maxPixels=1e9,
).getInfo()
print("newly cleared hectares (2020 -> 2022):", cleared_ha)

In [ ]:
m = geemap.Map()
m.centerObject(region, 11)
m.addLayer(ndvi1, {"min": 0, "max": 1, "palette": ["brown", "yellow", "green"]}, "NDVI 2020")
m.addLayer(cleared.selfMask(), {"palette": ["red"]}, "newly cleared")
m.addLayerControl()
m

In [ ]:
# Dynamic World 'trees' labels (the ground truth the U-Net / baseline is scored against)
from data.labels import get_trees_mask
from baseline.build_baseline import forest_mask as ee_forest_mask, iou as ee_iou, area_ha

trees = get_trees_mask(region, *DATE1)
baseline = ee_forest_mask(get_cloud_free_composite(region, *DATE1, CLOUD_PCT), 0.6)
print("baseline IoU vs DW trees:", round(ee_iou(baseline, trees, region), 4))
print("forest ha (NDVI):", round(area_ha(baseline, region), 1),
      " forest ha (DW):", round(area_ha(trees, region), 1))

In [ ]:
# Spot-check: DW trees vs the imagery + NDVI decision
m2 = geemap.Map()
m2.centerObject(region, 11)
m2.addLayer(get_cloud_free_composite(region, *DATE1, CLOUD_PCT),
            {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "S2 2020")
m2.addLayer(trees.selfMask(), {"palette": ["green"]}, "DW trees")
m2.addLayer(baseline.selfMask(), {"palette": ["yellow"]}, "NDVI > 0.6")
m2.addLayerControl()
m2

In [ ]:
# Local numpy path (used by src/evaluate.py on exported tiles) — runs without EE:
red = np.array([[0.05, 0.40], [0.10, 0.30]])
nir = np.array([[0.50, 0.50], [0.45, 0.20]])
true = np.array([[1, 1], [1, 0]], dtype=bool)

pred = ndvi_forest_mask(red, nir, threshold=0.6)
print("NDVI:\n", np.round(compute_ndvi(red, nir), 2))
print("baseline IoU:", round(iou(pred, true), 3))